# Pinecone

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [2]:
pip install pinecone-client -q

Note: you may need to restart the kernel to use updated packages.


In [9]:
pip install pinecone -q

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install --upgrade pinecone-client -q

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip show pinecone-client

Name: pinecone-client
Version: 6.0.0
Summary: Pinecone client (DEPRECATED)
Home-page: https://www.pinecone.io
Author: Pinecone Systems, Inc.
Author-email: support@pinecone.io
License: Apache-2.0
Location: C:\Users\Rinku\Desktop\python_genai\genai\Lib\site-packages
Requires: certifi, pinecone-plugin-interface, python-dateutil, typing-extensions, urllib3
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pinecone import Pinecone
pc= Pinecone()
pc.list_indexes()
index = pc.Index("my-q-a-index")
pc.describe_index('my-q-a-index')

Name:,my-q-a-index
Status:,Ready
Ready:,Yes
Deployment:,Managed (aws/us-east-1)
Host:,https://my-q-a-index-ik4gq08.svc.aped-4627-b74a.pinecone.io
Deletion Protection:,disabled
Schema fields:,1
Read capacity:,"ReadCapacityOnDemandResponse(status=ReadCapacityStatus(state='Ready', current_shards=None, current_replicas=None, error_message=None))"


# working with  pinecone inedexes

In [18]:
pc.list_indexes().names()

['my-q-a-index']

In [32]:
from pinecone import ServerlessSpec
index_name='langchain'
if index_name not in pc.list_indexes().names():
    print(f'create index: {index_name}')
    pc.create_index(
        name=index_name,
        dimension=7675,
        metric='cosine',
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )
    print('Index created!')
else:
    print(f'already index: {index_name}')

create index: langchain
Index created!


In [31]:
if index_name  in pc.list_indexes().names():
    print(f'deleting index: {index_name}')
    pc.delete_index(index_name)
    print('deleted!')
else:
    print(f'not exist index: {index_name}')

deleting index: langchain
deleted!


In [23]:
index = pc.Index("my-q-a-index")
index.describe_index_stats()

DescribeIndexStatsResponse(dimension=1024, total_vector_count=0, metric='cosine', namespaces=0)

# working with vector

In [4]:
import random

vectors = [
    [random.random() for _ in range(7675)]
    for _ in range(5)
]

ids = list('abcde')

index_name = 'langchain'
index = pc.index(index_name)

print(index.describe_index_stats())

index.upsert(vectors=list(zip(ids, vectors)))

DescribeIndexStatsResponse(dimension=7675, total_vector_count=5, metric='cosine', namespaces=1)


UpsertResponse(upserted_count=5)

In [25]:
index.upsert(vectors=[('c',[0.5]*7675)])

UpsertResponse(upserted_count=1)

In [26]:
index.fetch(ids=['c','d'])

FetchResponse(vectors={'c': Vector(id='c', values=[0.5, 0.5, 0.5, ...7672 more], sparse_values=None), 'd': Vector(id='d', values=[0.0244306475, 0.18543227, 0.695585549, ...7672 more], sparse_values=None)}, namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Tue, 29 Sep 2026 08:19:12 GMT', 'content-type': 'application/json', 'content-length': '121815', 'connection': 'keep-alive', 'x-pinecone-request-latency-ms': '220', 'x-envoy-upstream-service-time': '221', 'x-pinecone-response-duration-ms': '222', 'grpc-status': '0', 'server': 'envoy'}))

In [28]:
index.delete(ids=['b','c'])

In [29]:
index.describe_index_stats()

DescribeIndexStatsResponse(dimension=7675, total_vector_count=3, metric='cosine', namespaces=1)

In [30]:
index.fetch(ids=['x'])

FetchResponse(vectors={}, namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Tue, 29 Sep 2026 08:20:52 GMT', 'content-type': 'application/json', 'content-length': '69', 'connection': 'keep-alive', 'x-pinecone-request-latency-ms': '7', 'x-envoy-upstream-service-time': '8', 'x-pinecone-response-duration-ms': '9', 'grpc-status': '0', 'server': 'envoy'}))

In [33]:
# query
query_vector=[random.random() for _ in range(7675)]


In [39]:
index.query(
    vector=query_vector,
    top_k=3,
    include_values=False
)

QueryResponse(matches=[ScoredVector(id='b', score=0.752440274, values=[]), ScoredVector(id='e', score=0.750481725, values=[]), ScoredVector(id='d', score=0.749680877, values=[])], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Tue, 29 Sep 2026 08:24:44 GMT', 'content-type': 'application/json', 'content-length': '211', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '1', 'x-pinecone-request-latency-ms': '73', 'x-envoy-upstream-service-time': '48', 'x-pinecone-response-duration-ms': '75', 'grpc-status': '0', 'server': 'envoy'}))

# Namespace

In [ ]:
import random

vectors = [
    [random.random() for _ in range(7675)]
    for _ in range(5)
]

ids = list('abcde')

index_name = 'langchain'
index = pc.index(index_name)

print(index.describe_index_stats())

index.upsert(vectors=list(zip(ids, vectors)))

In [5]:
vectors = [
    [random.random() for _ in range(7675)]
    for _ in range(5)
]
ids = list('xyz')

index.upsert(vectors=list(zip(ids, vectors)), namespace='first-namespace')

UpsertResponse(upserted_count=3)

In [6]:
vectors = [
    [random.random() for _ in range(7675)]
    for _ in range(5)
]
ids = list('qwe')

index.upsert(vectors=list(zip(ids, vectors)), namespace='second-namespace')

UpsertResponse(upserted_count=3)

In [7]:
index.describe_index_stats()

DescribeIndexStatsResponse(dimension=7675, total_vector_count=11, metric='cosine', namespaces=3)

In [8]:
index.fetch(ids=['x'])

FetchResponse(vectors={}, namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Tue, 29 Sep 2026 08:48:25 GMT', 'content-type': 'application/json', 'content-length': '69', 'connection': 'keep-alive', 'x-pinecone-request-latency-ms': '41', 'x-envoy-upstream-service-time': '42', 'x-pinecone-response-duration-ms': '43', 'grpc-status': '0', 'server': 'envoy'}))

In [9]:
index.fetch(ids=['x'], namespace='first-namespace')

FetchResponse(vectors={'x': Vector(id='x', values=[0.61415261, 0.974161, 0.636861265, ...7672 more], sparse_values=None)}, namespace='first-namespace', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Tue, 29 Sep 2026 08:48:51 GMT', 'content-type': 'application/json', 'content-length': '91241', 'connection': 'keep-alive', 'x-pinecone-request-latency-ms': '75', 'x-envoy-upstream-service-time': '76', 'x-pinecone-response-duration-ms': '77', 'grpc-status': '0', 'server': 'envoy'}))